In [1]:
import csv
import random
from datetime import datetime, timedelta

# Set seed for reproducibility
random.seed(42)

# 1. EMISSION FACTORS LOOKUP (Real EPA / GHG Protocol Approximations)
# Values represent metric tons of CO2e per unit specified
emission_factors = [
    {"factor_id": "EF_001", "activity_type": "Diesel_Gallon", "ghg_protocol_scope": 1, "co2e_factor": 0.01021, "unit_of_measure": "Gallon"},
    {"factor_id": "EF_002", "activity_type": "Diesel_Liter", "ghg_protocol_scope": 1, "co2e_factor": 0.00268, "unit_of_measure": "Liter"},
    {"factor_id": "EF_003", "activity_type": "Electricity_kWh_US_Grid", "ghg_protocol_scope": 2, "co2e_factor": 0.00039, "unit_of_measure": "kWh"},
    {"factor_id": "EF_004", "activity_type": "Electricity_kWh_EU_Grid", "ghg_protocol_scope": 2, "co2e_factor": 0.00023, "unit_of_measure": "kWh"},
    {"factor_id": "EF_005", "activity_type": "Freight_Truck_Ton_Mile", "ghg_protocol_scope": 3, "co2e_factor": 0.00015, "unit_of_measure": "Ton-Mile"},
    {"factor_id": "EF_006", "activity_type": "Freight_Rail_Ton_Mile", "ghg_protocol_scope": 3, "co2e_factor": 0.00002, "unit_of_measure": "Ton-Mile"}
]

# 2. GENERATE FIELD OPERATIONS LOG (Scope 1)
# Edge cases: Mixed units (Gallons/Liters), missing values, negative anomalies, case variance
def generate_field_ops(num_rows=1500):
    farms = ["Iowa_North_02", "Illinois_East_11", "Mato_Grosso_S1", "Kyiv_Obl_04", "indiana_west_01"]
    crops = ["Corn", "Soybeans", "Wheat", "Canola", "Sunflowers"]
    equip = ["John Deere 8R", "Case IH Axial-Flow", "New Holland T7"]
    
    with open("field_operations_raw.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["log_id", "date", "farm_id", "crop_type", "equipment_type", "fuel_type", "fuel_consumed", "uom"])
        
        start_date = datetime(2025, 1, 1)
        for i in range(1, num_rows + 1):
            date_str = (start_date + timedelta(days=random.randint(0, 360))).strftime("%Y-%m-%d")
            farm = random.choice(farms)
            # Edge case: introduce mixed casing for farms to trigger standardizing transformations later
            if random.random() < 0.1: farm = farm.lower()
            
            fuel = round(random.uniform(10.0, 450.0), 2)
            uom = "Gallon" if "US" in farm or "Iowa" in farm or "Ill" in farm or "ind" in farm else "Liter"
            
            # Injecting explicit realistic anomalies for data cleaning practice
            if i % 150 == 0: fuel = ""          # Missing data edge case
            elif i % 200 == 0: fuel = -50.0     # Negative entry data corruption case
            
            writer.writerow([f"FLD_{i:04d}", date_str, farm, random.choice(crops), random.choice(equip), "Diesel", fuel, uom])

# 3. GENERATE FACILITY ENERGY LOG (Scope 2)
# Edge cases: Blank regions, mixed electricity structures
def generate_facility_energy(num_rows=500):
    facilities = [
        ("Pioneer_Seed_Plant_01", "North America"), ("Indianapolis_HQ", "North America"),
        ("Rochelle_Production", "North America"), ("Toulouse_Research", "Europe"),
        ("Eschbach_Station", "Europe"), ("Cuiaba_Hub", "Latin America")
    ]
    
    with open("facility_energy_raw.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["invoice_id", "billing_period_start", "facility_id", "region", "utility_provider", "energy_type", "consumption_kwh"])
        
        start_date = datetime(2025, 1, 1)
        for i in range(1, num_rows + 1):
            date_str = (start_date + timedelta(days=random.randint(0, 11) * 30)).strftime("%Y-%m-%d")
            fac, region = random.choice(facilities)
            
            # Edge case: Missing region fields to simulate fragmented ERP data systems
            if random.random() < 0.08: region = ""
                
            kwh = round(random.uniform(5000.0, 85000.0), 2)
            writer.writerow([f"INV_{i:04d}", date_str, fac, region, "GlobalUtility Corp", "Electricity", kwh])

# 4. GENERATE THIRD-PARTY LOGISTICS LOG (Scope 3)
# Edge cases: Weight unit fragmentation requiring preprocessing calculations
def generate_logistics(num_rows=2000):
    hubs = ["Des_Moines_Hub", "Rotterdam_Port", "Santos_Terminal", "Calgary_Depot", "Indy_Distribution"]
    modes = ["Heavy Duty Truck", "Rail"]
    uoms = ["lbs", "kg", "MT"]
    
    with open("logistics_shipping_raw.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["shipment_id", "shipment_date", "origin_facility", "destination_hub", "transport_mode", "cargo_weight", "weight_uom", "distance_miles"])
        
        start_date = datetime(2025, 1, 1)
        for i in range(1, num_rows + 1):
            date_str = (start_date + timedelta(days=random.randint(0, 360))).strftime("%Y-%m-%d")
            weight = round(random.uniform(500.0, 45000.0), 2)
            uom = random.choice(uoms)
            dist = round(random.uniform(50.0, 1200.0), 1)
            
            writer.writerow([f"SHP_{i:04d}", date_str, "Production_Plant_Alpha", random.choice(hubs), random.choice(modes), weight, uom, dist])

# 5. WRITE LOOKUP TABLE
def write_lookup():
    with open("emission_factors_lookup.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["factor_id", "activity_type", "ghg_protocol_scope", "co2e_factor", "unit_of_measure"])
        writer.writeheader()
        writer.writerows(emission_factors)

if __name__ == "__main__":
    print("Initializing realistic agribusiness ESG datasets...")
    generate_field_ops()
    generate_facility_energy()
    generate_logistics()
    write_lookup()
    print("Success! Generated raw datasets containing realistic edge cases.")


Initializing realistic agribusiness ESG datasets...
Success! Generated raw datasets containing realistic edge cases.
